In [1]:
import os
import torch
import warnings
import chunking as C
from tqdm import tqdm
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from transformer_vae import TVAE, vae_loss
from cifar_eval_data import load_cifar_eval
from torch.utils.data import DataLoader, Dataset
from constants import SEED, device, TOKENS_PER_SEQ, IMG_PATH, ZOO_PATH
from reconstruction import interp_to_state_dict, eval_state_dict, reconstruct_model, evaluate_grouped, evaluate_reconstruction
from dataset import ResZoo, summon_res_zoo
warnings.filterwarnings("ignore")

In [12]:
BETA = 3e-6
tvae_model = TVAE()
tvae_model.load_state_dict(torch.load(f'./res_models/tvae_r1_beta{BETA}.pt', weights_only=True)['state_dict'])
tvae_model = tvae_model.to(device)

In [13]:
(train_dataset_cifar, test_dataset_cifar, full_test_loader, zoo_train_feed, zoo_test_feed) = load_cifar_eval('./res_data/')

FIRST_EXPERT_ID = 3
SECOND_EXPERT_ID = 4
FIRST_MODEL_ID = 'split3_seed1246_ep40'
SECOND_MODEL_ID = 'split4_seed1246_ep40'

'''separate datasets, one per expert'''
ds_one = ResZoo(root_dir=ZOO_PATH, model_ids=[FIRST_MODEL_ID])
ds_two = ResZoo(root_dir=ZOO_PATH, model_ids=[SECOND_MODEL_ID])
loader_one = DataLoader(ds_one, batch_size=64, shuffle=False)
loader_two = DataLoader(ds_two, batch_size=64, shuffle=False)

ck1 = torch.load('res_models/expert3_seed1246/ep040.pt', map_location='cpu')
ck2 = torch.load('res_models/expert4_seed1246/ep040.pt', map_location='cpu')

'''backbone as the reconstruction target, its head works across all 100 classes.
using an expert's head would cap the merge at ~20% by construction'''
ck_bb = torch.load('res_models/backbone_a.pt', map_location='cpu')

'''recalibrate over both experts' classes, the merged model should span both'''
recal_idx = zoo_train_feed[FIRST_EXPERT_ID] + zoo_train_feed[SECOND_EXPERT_ID]
recal_subset = torch.utils.data.Subset(train_dataset_cifar, recal_idx)
recal_loader = DataLoader(recal_subset, batch_size=128, shuffle=True, num_workers=2)

'''sanity check, the two manifests must line up for chunkwise interpolation'''
assert ds_one.meta_list[0].n_chunks_total == ds_two.meta_list[0].n_chunks_total
assert len(ds_one) == len(ds_two)

In [14]:
@torch.no_grad()
def merge_two(model, loader_a, loader_b, lam):
    '''latent space merge: z = (1-lam)*z_a + lam*z_b, decoded chunk by chunk. lam=0 gives expert a, lam=1 gives expert b'''
    model.eval()
    outs = []
    for batch_a, batch_b in zip(loader_a, loader_b):
        depth = batch_a['depth'].to(device, non_blocking=True)
        stage = batch_a['stage'].to(device, non_blocking=True)

        '''use mu, not a sample, so the merge is deterministic'''
        mu_a, _ = model.encode(batch_a['chunks'].to(device), depth, stage)
        mu_b, _ = model.encode(batch_b['chunks'].to(device), depth, stage)

        z = (1 - lam) * mu_a + lam * mu_b
        outs.append(model.decode(z, depth, stage).cpu())

    return torch.concat(outs, dim=0)


def merge_weight_space(sd_a, sd_b, keys, lam):
    '''baseline: plain linear interpolation on the raw weights,
    same keys the vae handles so the comparison is like for like'''
    out = {k: v.clone() for k, v in sd_a.items()}
    for k in keys:
        out[k] = (1 - lam) * sd_a[k].float() + lam * sd_b[k].float()
    return out
    

In [15]:
'''keys the vae actually reconstructs, so weight space merges the same 19 tensors'''
MERGE_KEYS = [lm.key for lm in ds_one.meta_list[0].layers]

lambdas = torch.linspace(0.0, 1.0, 11)
rows = []

for lam in tqdm(lambdas):
    lam = lam.item()

    '''latent space, decoded onto the backbone so the head spans all 100 classes'''
    out = merge_two(tvae_model, loader_one, loader_two, lam)
    sd_lat = interp_to_state_dict(out, ds_one, ck_bb['state_dict'])
    own3_l, _, all_l = eval_state_dict(sd_lat, FIRST_EXPERT_ID, recal_loader, full_test_loader)
    own4_l, _, _     = eval_state_dict(sd_lat, SECOND_EXPERT_ID, recal_loader, full_test_loader)

    '''weight space baseline, same lam, same recalibration'''
    sd_w = merge_weight_space(ck1['state_dict'], ck2['state_dict'], MERGE_KEYS, lam)
    for k, v in ck_bb['state_dict'].items():
        if k not in MERGE_KEYS:
            sd_w[k] = v.clone()
    own3_w, _, all_w = eval_state_dict(sd_w, FIRST_EXPERT_ID, recal_loader, full_test_loader)
    own4_w, _, _     = eval_state_dict(sd_w, SECOND_EXPERT_ID, recal_loader, full_test_loader)

    rows.append((lam, own3_l, own4_l, all_l, own3_w, own4_w, all_w))
    print(f'lam {lam:.2f} | latent e3 {own3_l:6.2f} e4 {own4_l:6.2f} all {all_l:6.2f} '
          f'| weight e3 {own3_w:6.2f} e4 {own4_w:6.2f} all {all_w:6.2f}')

  9%|█████████████                                                                                                                                  | 1/11 [00:21<03:35, 21.51s/it]

lam 0.00 | latent e3  82.70 e4   8.80 all  26.30 | weight e3  82.60 e4   6.60 all  23.20


 18%|██████████████████████████                                                                                                                     | 2/11 [00:42<03:11, 21.33s/it]

lam 0.10 | latent e3  83.15 e4  22.00 all  32.26 | weight e3  82.75 e4  18.80 all  29.09


 27%|███████████████████████████████████████                                                                                                        | 3/11 [01:03<02:50, 21.30s/it]

...

lam 0.80 | latent e3  32.35 e4  88.20 all  33.68 | weight e3  31.45 e4  88.70 all  31.81


 91%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████             | 10/11 [03:32<00:21, 21.24s/it]

lam 0.90 | latent e3  17.45 e4  88.50 all  27.80 | weight e3  16.25 e4  88.95 all  26.07


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [03:53<00:00, 21.27s/it]

lam 1.00 | latent e3   7.70 e4  88.40 all  23.17 | weight e3   6.20 e4  89.00 all  21.82
